# PolyChain — v29 Training (Colab T4)

Trains all models on T4 GPU, saves predictions + checkpoints to Google Drive.
Upload saved files to a lightweight Kaggle notebook for ensemble + submission.

In [ ]:
# Cell 1: Mount Drive + install dependencies
import sys, os, subprocess, shutil, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path("/content/drive/MyDrive/Poly/v29_predictions")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive output: {DRIVE_DIR}")

WORK_DIR = Path("/content")
os.chdir(WORK_DIR)

def detect_gpu():
    try:
        out = subprocess.check_output(
            "nvidia-smi --query-gpu=name --format=csv,noheader",
            shell=True, timeout=10
        ).decode().strip()
        return out[:60]
    except Exception:
        return "CPU"

gpu = detect_gpu()
print(f"GPU: {gpu}")

def pip_install(pkg, extra_args=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if extra_args:
        cmd.extend(extra_args)
    cmd.append(pkg)
    subprocess.run(cmd, capture_output=True)

# PyTorch (CUDA 12.x for T4)
try:
    import torch
    print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
except ImportError:
    print("Installing PyTorch...")
    pip_install("torch==2.6.0+cu124",
                ["--index-url", "https://download.pytorch.org/whl/cu124"])
    import torch
    print(f"PyTorch {torch.__version__}")

# PyG
if subprocess.run([sys.executable, "-c", "import torch_geometric"],
                  capture_output=True).returncode != 0:
    print("Installing PyTorch Geometric...")
    pip_install("torch_geometric")

# Other deps
for pkg in ["rdkit", "xgboost", "lightgbm", "catboost", "optuna",
            "scikit-learn", "pandas", "numpy", "pyyaml", "pyarrow",
            "kagglehub"]:
    if subprocess.run([sys.executable, "-c", f"import {pkg.replace('-', '_')}"],
                      capture_output=True).returncode != 0:
        pip_install(pkg)

print("All dependencies ready!")

In [ ]:
# Cell 2: Clone repo + copy code
REPO_URL = "https://github.com/NotShubham1112/Poly.git"
BRANCH = "main"

os.chdir(WORK_DIR)
REPO_DIR = WORK_DIR / "Poly"

if REPO_DIR.exists():
    os.chdir(str(REPO_DIR))
    subprocess.check_call(["git", "checkout", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(["git", "pull", "origin", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.chdir(str(WORK_DIR))
else:
    subprocess.check_call([
        "git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

SRC = REPO_DIR / "polymer_competition"
for item in os.listdir(str(SRC)):
    src_path = SRC / item
    dst_path = WORK_DIR / item
    if dst_path.exists():
        if item == "data":
            continue  # Preserve generated features across re-runs
        if dst_path.is_dir():
            shutil.rmtree(str(dst_path))
        else:
            os.remove(str(dst_path))
    if src_path.is_dir():
        shutil.copytree(str(src_path), str(dst_path))
    else:
        shutil.copy2(str(src_path), str(dst_path))

print(f"Code ready at {WORK_DIR}")

In [ ]:
# Cell 3: Download competition data via kagglehub
import kagglehub

data_dir = WORK_DIR / "data"
train_csv = data_dir / "train.csv"
test_csv = data_dir / "test.csv"

if not train_csv.exists() or not test_csv.exists():
    print("Downloading competition data via kagglehub...")
    try:
        path = kagglehub.competition_download('aisehack-2-0')
        import shutil
        for f in ["train.csv", "test.csv"]:
            src = Path(path) / f
            if src.exists():
                shutil.copy(str(src), str(data_dir / f))
        print(f"Data downloaded from: {path}")
    except Exception as e:
        print(f"kagglehub download failed: {e}")
        if not train_csv.exists():
            raise RuntimeError(
                "No training data found. Run `kaggle competitions download` manually "
                "or place train.csv/test.csv in the data/ directory."
            )
else:
    print("Data already present, skipping download.")

print(f"train.csv: {train_csv.stat().st_size if train_csv.exists() else 0} bytes")
print(f"test.csv:  {test_csv.stat().st_size if test_csv.exists() else 0} bytes")

In [ ]:
# Cell 4: Config setup (v29, experiment tracking)
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["experiment"]["version"] = "v29"
cfg["experiment"]["tracking"] = False  # Disable manifest (causes git stderr on Kaggle)

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Experiment version: {cfg["experiment"]["version"]}')
print(f'Paths data_dir: {cfg["paths"]["data_dir"]}')

In [ ]:
# Cell 5: Build features + CV splits (no GNN embeddings yet)
!python -m data.generate_splits --config config.yaml --target tg
!python -m data.generate_splits --config config.yaml --target egc
!python -m features.build_features --config config.yaml
print("Features and CV splits ready!")

In [ ]:
# Cell 6: Train tree models 5-fold, both targets
# All bug fixes baked in: direct --n_seeds (not --n-seeds), --target flag
import subprocess, sys

WORK_DIR_STR = str(WORK_DIR)

def train_model(model_type, target, fold, config="config.yaml", **kwargs):
    cmd = [sys.executable, "-m", "training.train",
           "--model_type", model_type,
           "--fold", str(fold),
           "--target", target,
           "--config", config]
    for k, v in kwargs.items():
        cmd.extend([f"--{k}", str(v)])  # Direct --n_seeds, NOT --n-seeds
    r = subprocess.run(cmd, cwd=WORK_DIR_STR)
    if r.returncode != 0:
        print(f"FAILED: {model_type} {target} fold {fold}")
    return r.returncode == 0

tree_models = ["xgb", "lgb", "catboost", "rf"]
for model in tree_models:
    for target in ["tg", "egc"]:
        for fold in range(5):
            train_model(model, target, fold)
print("Tree models complete!")

In [ ]:
# Cell 7: Train GNNs + auto-extract embeddings, 5-fold both targets
gnn_models = ["gcn", "gat", "mpnn"]
for model in gnn_models:
    for target in ["tg", "egc"]:
        for fold in range(5):
            train_model(model, target, fold)
print("GNN training + embedding extraction complete!")

In [ ]:
# Cell 8: Rebuild features with GNN embeddings
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["features"]["use_embeddings"] = True
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

!python -m features.build_features --config config.yaml
print("Features rebuilt with GNN embeddings!")

In [ ]:
# Cell 9: Train MLP 5-seed ensemble with Huber loss
for target in ["tg", "egc"]:
    for fold in range(5):
        train_model("mlp", target, fold, n_seeds=5, loss="huber")
print("MLP multi-seed training complete!")

In [ ]:
# Cell 10: Multi-task training 80 epochs
# MultiTaskModel uses LayerNorm (not BatchNorm) so batch_size=1 is safe
!python run_multitask.py --epochs 80
print("Multi-task training complete!")

In [ ]:
# Cell 11: Save predictions + models to Google Drive
import pickle, shutil

# Copy prediction files
pred_dir = WORK_DIR / "predictions"
if pred_dir.exists():
    for f in pred_dir.glob("*"):
        shutil.copy2(str(f), str(DRIVE_DIR / f.name))
    print(f"Copied {len(list(pred_dir.glob('*')))} prediction files to Drive")

# Copy model checkpoints
ckpt_dir = WORK_DIR / "checkpoints"
if ckpt_dir.exists() and ckpt_dir.is_dir():
    drive_ckpt = DRIVE_DIR / "checkpoints"
    drive_ckpt.mkdir(parents=True, exist_ok=True)
    for f in ckpt_dir.glob("*"):
        shutil.copy2(str(f), str(drive_ckpt / f.name))
    print(f"Copied checkpoints to Drive")

print(f"\nAll saved to: {DRIVE_DIR}")
print("Download these files and upload to your Kaggle notebook for ensemble + submission.")

In [ ]:
# Cell 12: Summary — show per-model R2 scores
import json, numpy as np
from collections import defaultdict

manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    results = defaultdict(lambda: defaultdict(list))
    for entry in data:
        if entry.get("status") == "completed":
            model = entry.get("model_type", "?")
            target = entry.get("target", "?")
            score = entry.get("score", 0)
            if isinstance(score, (list, tuple)):
                score = sum(score) / len(score)
            results[model][target].append(score)

    print(f"{'Model':<20} {'TG R2':>10} {'EGC R2':>10} {'Mean R2':>10}")
    print("-" * 52)
    for model in sorted(results.keys()):
        tg_scores = results[model].get("tg", [0])
        egc_scores = results[model].get("egc", [0])
        tg_mean = sum(tg_scores) / len(tg_scores) if tg_scores else 0
        egc_mean = sum(egc_scores) / len(egc_scores) if egc_scores else 0
        mean_r2 = (tg_mean + egc_mean) / 2
        print(f"{model:<20} {tg_mean:>10.4f} {egc_mean:>10.4f} {mean_r2:>10.4f}")

    tg_models = [m for m in results if "tg" in results[m]]
    egc_models = [m for m in results if "egc" in results[m]]
    if tg_models:
        tg_best = max(max(results[m]["tg"]) for m in tg_models)
        egc_best = max(max(results[m]["egc"]) for m in egc_models)
        print(f"\nBest single fold TG: {tg_best:.4f}")
        print(f"Best single fold EGC: {egc_best:.4f}")
        print(f"Mean of best: {(tg_best + egc_best) / 2:.4f}")
else:
    print("No experiment manifest found — models may still be training.")